In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
orig_cfa_res = pd.read_csv('../data/cfa/orig_cfa_res.csv')

In [ ]:
orig_cfa_res

In [ ]:
palette = sns.color_palette('Pastel1', 8)

In [ ]:
palette

In [ ]:
# six categories now: Fails Configural / Configural / Thresholds / Metric / Scalar / Strict
colors = [palette[0], palette[5], palette[6], palette[4], palette[1], palette[2]]

In [ ]:
# Highest invariance level PASSED per scale x pair, computed explicitly
# from the p-values. (The old version counted null columns, which broke
# when the measEq pipeline added the thresholds level, and it credited the
# last level as passed merely for having been tested.)
# Levels are gated: a level was only tested if the one below passed, so a
# level counts as passed iff its p >= .05. Configural is special: it can
# also pass via the secondary fit criteria (p < .05 but the ladder
# continued), which shows up as a non-null thresholds p.
level_cols = ['pconfig', 'pthresholds', 'pmetric', 'pscalar', 'pstrict']


def highest_level_passed(row):
    config_passed = (
        (pd.notna(row['pconfig']) and row['pconfig'] >= 0.05)
        or pd.notna(row['pthresholds'])  # secondary-criteria pass
    )
    if not config_passed:
        return 0  # fails configural
    level = 1
    for col in level_cols[1:]:
        if pd.notna(row[col]) and row[col] >= 0.05:
            level += 1
        else:
            break
    return level


orig_cfa_res['level_passed'] = orig_cfa_res.apply(highest_level_passed, axis=1)
to_plot = orig_cfa_res.pivot(index='scale', columns='pair', values='level_passed')
to_plot = to_plot.reset_index()
to_plot['scale'] = to_plot.scale.str.replace('_', ' ').str.title()
to_plot['scale'] = to_plot.scale.replace('Shame Guilt', 'Shame/Guilt')
to_plot['scale'] = to_plot.scale.replace('Well Being', 'Well-Being')
to_plot = to_plot.rename(columns={'scale':'Scale'})
to_plot = to_plot.set_index('Scale')
to_plot.columns.name = 'Data Pairs'
to_plot = to_plot.rename(columns={'GP_EN':'GenPop\nEnriched', 'V_EN':'OG Valid\nEnriched', 'V_GP':'OG Valid\nGenPop'})

In [ ]:
sns.set_context('poster')
# spread level 0..5 onto odd values 1..11 so each category sits in the
# middle of its color band
to_plot_vals = to_plot * 2 + 1
fig, ax = plt.subplots(1, figsize=(5,10), dpi=250)
sns.heatmap(to_plot_vals, cmap=colors, vmax=12, vmin=0)
colorbar = ax.collections[0].colorbar
colorbar.set_ticks([1, 3, 5, 7, 9, 11])
colorbar.set_ticklabels(['Fails Configural', 'Configural', 'Thresholds',
                         'Metric', 'Scalar', 'Strict'])
colorbar.set_label('Invariance Level')
ax.xaxis.set_label_coords(0.5, -0.23)
